In [1]:
import spacy

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "job_dataset.csv"

# carrega dados de um dataset de trabalhos na area de TI
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles", file_path,)

In [3]:
df.head()

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,0-1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,0-1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


In [4]:
df.columns
# mostra as colunas no dataset

Index(['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills',
       'Responsibilities', 'Keywords'],
      dtype='object')

In [5]:
print(df[["Title", "ExperienceLevel", "YearsOfExperience", "Skills"]].head())
# primeiras 5 linhas das colunas

            Title ExperienceLevel YearsOfExperience  \
0  .NET Developer         Fresher               0-1   
1  .NET Developer         Fresher               0-1   
2  .NET Developer         Fresher               0-1   
3  .NET Developer         Fresher               0-1   
4  .NET Developer         Fresher               0-1   

                                              Skills  
0  C#; VB.NET basics; .NET Framework; .NET Core f...  
1  C#; .NET Framework basics; ASP.NET; Razor; HTM...  
2  C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...  
3  C#; .NET Framework; ASP.NET basics; SQL Server...  
4  C#; ASP.NET; MVC; Entity Framework basics; SQL...  


In [6]:
print(df["Skills"].iloc[0])

C#; VB.NET basics; .NET Framework; .NET Core fundamentals; ASP.NET; MVC; HTML; CSS; JavaScript basics; SQL Server; Entity Framework basics; LINQ; Visual Studio; Git; Unit Testing basics


In [7]:
# como algumas colunas possuem registros separados com ; e espacos, criei uma funcao para separar cada string e devolver uma lista python ja formatada
def sep_lista(texto):
    return [s.strip() for s in texto.split(";")] # para cada item separa pelo ; e os espacos sobrando

df["SkillsList"] = df["Skills"].apply(sep_lista)
# para cada item na coluna skills, aplica a funcao que separa
df["KeywordsList"] = df["Keywords"].apply(sep_lista)
# faco o mesmo para a coluna keywords, que tras os nomes das tecnologias
print(df["SkillsList"].iloc[0])
print(df["KeywordsList"].iloc[0])

['C#', 'VB.NET basics', '.NET Framework', '.NET Core fundamentals', 'ASP.NET', 'MVC', 'HTML', 'CSS', 'JavaScript basics', 'SQL Server', 'Entity Framework basics', 'LINQ', 'Visual Studio', 'Git', 'Unit Testing basics']
['.NET', 'C#', 'ASP.NET MVC', 'Entity Framework', 'SQL Server', 'LINQ', 'Visual Studio', 'Unit Testing']


In [8]:
all_terms = []
for skill_list in df["SkillsList"]:
    # para cada lista de skills dentro da coluna adiciona a lista, mas com todos os termos
    all_terms.extend(skill_list)
for keyword_list in df["KeywordsList"]:
    # faco o mesmo com keywords, juntando tudo numa unica lista de termos
    all_terms.extend(keyword_list)

print(len(all_terms))
# quantidade de termos totais (skills + keywords, com repeticao)
print(all_terms[:20])
# mostra os 20 primeiros termos

20832
['C#', 'VB.NET basics', '.NET Framework', '.NET Core fundamentals', 'ASP.NET', 'MVC', 'HTML', 'CSS', 'JavaScript basics', 'SQL Server', 'Entity Framework basics', 'LINQ', 'Visual Studio', 'Git', 'Unit Testing basics', 'C#', '.NET Framework basics', 'ASP.NET', 'Razor', 'HTML']


In [9]:
unique_terms = list(set(all_terms))
# deixa somente termos unicos, tirando as duplicadas

print(len(unique_terms))
# tamanho de termos unicos
print(sorted(unique_terms)[:20])

2570
['.NET', '.NET Core', '.NET Core basics', '.NET Core fundamentals', '.NET Framework', '.NET Framework basics', '3D Design', '3D Design Basics', '3D Graphics', '3D Modeling', '3D modeling and animation expert', '3D modeling and graphics', '3D modeling basics', '3D modeling fundamentals', '3D modeling/animation advanced', '3D modeling/animation expert', 'A/B Testing', 'A/B testing', 'AI', 'AI Analytics']


In [ ]:
sufixos = [
    " basics", " fundamentals", " advanced", " expert", " proficiency",
    " mastery", " basic", " advanced skills"
]

def limpar_termo(termo):
    # para cada termo, deixo em minuscula para comparar e verifico se termina com algum dos sufixos
    # .NET tem muitos sufixos, por isso vou resumir tudo pra .NET
    termo_lower = termo.lower()
    for sufixo in sufixos:
        if termo_lower.endswith(sufixo):
            return termo[: -len(sufixo)].strip()
            # tira o sufixo da string
    return termo

cleaned_terms = [limpar_termo(t) for t in unique_terms]
# chama a funcao de limpeza pra cada termo

seen = set()
gazetteer = []
for termo in cleaned_terms:
    # remove as duplicadas que sobraram depois da limpeza, ignorando maiuscula/minuscula
    key = termo.lower()
    if key not in seen:
        seen.add(key)
        gazetteer.append(termo)# adiciona o termo a lista

print(len(gazetteer))
print(sorted(gazetteer)[:20])

2083
['.NET', '.NET Core', '.NET Framework', '3D Design', '3D Graphics', '3D Modeling', '3D modeling and animation', '3D modeling and graphics', '3D modeling/animation', 'A/B testing', 'AI', 'AI Analytics', 'AI Dashboards', 'AI Design Tools', 'AI Ethics', 'AI Frameworks', 'AI Marketing Tools', 'AI Model Optimization', 'AI SEO Tools', 'AI Tools']


In [11]:
# alguns termos aparecem no texto corrido de forma abreviada e nao batem com a lista oficial
# o texto diz "AI models" mas a skill formal e "Machine Learning"
# entao adiciono essas variacoes manualmente na lista, do mesmo jeito que foi feito no curso com o "S&P 500"
sinonimos = [
    "AI models", "ML models", "AI/ML models", "ML pipelines",
    "deep learning models", "ML algorithms", "AI development",
    "AI pipelines", "AI platforms", "version control",
]
gazetteer.extend(sinonimos)
print(len(gazetteer))

2093


In [ ]:
nlp = spacy.blank("en")
# cria um pipe novo com linguagem ingles
ruler = nlp.add_pipe("entity_ruler")
# regra e adicionada a pipeline

patterns = []
for termo in gazetteer:
    patterns.append({"label": "TECH", "pattern": termo})
    # para cada termo atribui o label TECH, e e adicionado como dicionario na lista

ruler.add_patterns(patterns)
# adiciona a regra adiciona ao pipe a lista de dicionarios que são as regras
print(len(patterns))

2093


In [13]:
texto_teste = df["Responsibilities"].iloc[0]
# pegamos um texto de teste no dataset
print(texto_teste)

# transformamos em doc para ver como funciona a busca com base nas regras adicionadas
doc = nlp(texto_teste)
for ent in doc.ents:
    print(ent.text, ent.label_)

Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; Support team in building ASP.NET MVC web applications; Write basic SQL queries and work with Entity Framework; Collaborate with peers to solve issues; Participate in code reviews for learning; Follow best practices in coding; Work with version control (Git)
.NET Framework TECH
ASP.NET MVC TECH
SQL queries TECH
Entity Framework TECH
version control TECH
Git TECH


In [14]:
for i in range(5):
    # roda o loop 5 vezes testando as regras inseridas para 5 textos diferentes
    texto = df["Responsibilities"].iloc[i]
    doc = nlp(texto)
    print(f"Vaga {i} - {df['Title'].iloc[i]} ")
    for ent in doc.ents:
        print(" ", ent.text, ent.label_)
    print()

Vaga 0 - .NET Developer 
  .NET Framework TECH
  ASP.NET MVC TECH
  SQL queries TECH
  Entity Framework TECH
  version control TECH
  Git TECH

Vaga 1 - .NET Developer 
  C# TECH
  ASP.NET MVC TECH
  Razor TECH
  LINQ TECH

Vaga 2 - .NET Developer 
  MVC TECH
  version control TECH

Vaga 3 - .NET Developer 
  UI TECH
  LINQ TECH

Vaga 4 - .NET Developer 
  ASP.NET TECH
  SQL Server TECH
  UI TECH



In [15]:
contagens = []
for i in range(len(df)):
    # roda o pipeline em todas as vagas do dataset
    texto = df["Responsibilities"].iloc[i]
    doc = nlp(texto)
    contagens.append(len(doc.ents))

df["QtdTechEncontradas"] = contagens
print(df["QtdTechEncontradas"].describe())
# estatisticas gerais de quantas techs foram encontradas por vaga

count    1068.000000
mean        2.008427
std         1.867927
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000
Name: QtdTechEncontradas, dtype: float64
